In [5]:
import pandas as pd
import numpy as np

# =========================
# 1. Weather
# =========================

weather = pd.read_csv(
    "weather data and radiation.csv"
)

print("\n========== WEATHER ==========")

print(weather.head())

print("\nshape:")
print(weather.shape)

print("\ncolumns:")
print(weather.columns.tolist())

print("\nmissing:")
print(weather.isna().sum())

# =========================
# 2. UVI
# =========================

uvi = pd.read_csv(
    "uv_5min.txt",
    sep=r"\s+",
    skiprows=1,           # 跳过表头
    names=["time", "UVI"]
)

uvi["time"] = pd.to_datetime(
    uvi["time"].astype(str),
    format="%Y%m%d%H%M"
)

# =========================
# 3. Clear Sky
# =========================

clear = pd.read_csv(
    "clear_sky_ghi_hkt.csv"
)

print("\n========== CLEAR SKY ==========")

print(clear.head())

print("\nshape:")
print(clear.shape)

print("\ncolumns:")
print(clear.columns.tolist())

print("\nmissing:")
print(clear.isna().sum())


========== WEATHER ==========
            time  wind_d  windspd  windspd_max  temp  rainfall  GHI  DNI  DHI  \
0  2020/1/1 0:01    89.0      3.0          3.7  17.3       0.0  0.0  0.0  0.0   
1  2020/1/1 0:02    89.0      2.6          3.9  17.4       0.0  0.0  0.0  0.0   
2  2020/1/1 0:03    90.0      3.9          4.6  17.4       0.0  0.0  0.0  0.0   
3  2020/1/1 0:04    93.0      3.1          4.5  17.3       0.0  0.0  0.0  0.0   
4  2020/1/1 0:05    98.0      2.0          3.6  17.3       0.0  0.0  0.0  0.0   

   uva  
0  0.0  
1  0.0  
2  0.0  
3  0.0  
4  0.0  

shape:
(1048575, 10)

columns:
['time', 'wind_d', 'windspd', 'windspd_max', 'temp', 'rainfall', 'GHI', 'DNI', 'DHI', 'uva']

missing:
time              0
wind_d         7756
windspd        7756
windspd_max    7757
temp           5680
rainfall       5065
GHI             379
DNI             552
DHI             687
uva             316
dtype: int64

========== CLEAR SKY ==========
                  time  Clear sky GHI
0  2020-0

In [6]:
weather['time']=pd.to_datetime(weather['time'])
clear['time']=pd.to_datetime(clear['time'])

print(weather['time'].min())
print(weather['time'].max())

print(clear['time'].min())
print(clear['time'].max())

2020-01-01 00:01:00
2021-12-29 04:15:00
2020-01-01 09:00:00
2023-01-02 08:00:00


In [7]:
weather.set_index('time', inplace=True)

hourly = weather.resample('1H').agg({
    'temp':'mean',
    'rainfall':'sum',
    'windspd':'mean',
    'windspd_max':'max',
    'wind_d':'mean',
    'GHI':'mean',
    'DNI':'mean',
    'DHI':'mean',
    'uva':'mean'
})

print(hourly.shape)
print(hourly.head())
print(hourly.isna().sum())

C:\Users\apple\AppData\Local\Temp\ipykernel_13920\3410616590.py:3: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly = weather.resample('1H').agg({


(17477, 9)
                          temp  rainfall   windspd  windspd_max     wind_d  \
time                                                                         
2020-01-01 00:00:00  17.318644       0.0  3.006780          7.2  85.593220   
2020-01-01 01:00:00  17.208333       0.0  2.673333          6.7  83.300000   
2020-01-01 02:00:00  16.926667       0.0  3.315000          7.7  90.950000   
2020-01-01 03:00:00  16.696667       0.0  2.481667          6.1  84.833333   
2020-01-01 04:00:00  16.586667       0.0  2.223333          5.4  91.600000   

                     GHI  DNI  DHI  uva  
time                                     
2020-01-01 00:00:00  0.0  0.0  0.0  0.0  
2020-01-01 01:00:00  0.0  0.0  0.0  0.0  
2020-01-01 02:00:00  0.0  0.0  0.0  0.0  
2020-01-01 03:00:00  0.0  0.0  0.0  0.0  
2020-01-01 04:00:00  0.0  0.0  0.0  0.0  
temp            79
rainfall         0
windspd        115
windspd_max    115
wind_d         115
GHI              4
DNI              4
DHI            

In [9]:
import pandas as pd
import numpy as np

# ============================================================
# 1. 读取 Weather 数据
# ============================================================

print("读取 Weather 数据...")

weather = pd.read_csv("weather data and radiation.csv")

weather["time"] = pd.to_datetime(weather["time"])

weather = weather.set_index("time")

# 重采样为小时数据
weather_hourly = weather.resample("1H").agg({
    "temp": "mean",
    "rainfall": "sum",
    "windspd": "mean",
    "windspd_max": "max",
    "wind_d": "mean",
    "GHI": "mean",
    "DNI": "mean",
    "DHI": "mean",
    "uva": "mean"
})

print("Weather Hourly Shape:", weather_hourly.shape)

# ============================================================
# 2. Weather 缺失值处理
# ============================================================

continuous_cols = [
    "temp",
    "windspd",
    "windspd_max",
    "wind_d",
    "GHI",
    "DNI",
    "DHI",
    "uva"
]

weather_hourly[continuous_cols] = (
    weather_hourly[continuous_cols]
    .interpolate(
        method="linear",
        limit_direction="both"
    )
)

weather_hourly["rainfall"] = (
    weather_hourly["rainfall"]
    .fillna(0)
)

# ============================================================
# 3. 读取 UVI 数据
# ============================================================

print("读取 UVI 数据...")

uvi = pd.read_csv(
    "uv_5min.txt",
    sep=r"\s+",
    skiprows=1,
    names=["time", "UVI"]
)

uvi["time"] = pd.to_datetime(
    uvi["time"].astype(str),
    format="%Y%m%d%H%M"
)

uvi = uvi.set_index("time")

uvi_hourly = (
    uvi
    .resample("1H")
    .mean()
)

print("UVI Hourly Shape:", uvi_hourly.shape)

# ============================================================
# 4. 读取 Clear Sky 数据
# ============================================================

print("读取 Clear Sky 数据...")

clear = pd.read_csv("clear_sky_ghi_hkt.csv")

clear["time"] = pd.to_datetime(clear["time"])

clear = clear.set_index("time")

clear_hourly = clear.resample("1H").mean()

# 前8小时缺失补0
clear_hourly["Clear sky GHI"] = (
    clear_hourly["Clear sky GHI"]
    .fillna(0)
)

print("Clear Sky Shape:", clear_hourly.shape)

# ============================================================
# 5. 合并数据
# ============================================================

print("开始合并数据...")

df = weather_hourly.join(
    uvi_hourly,
    how="inner"
)

df = df.join(
    clear_hourly,
    how="left"
)

df["Clear sky GHI"] = (
    df["Clear sky GHI"]
    .fillna(0)
)

# ============================================================
# 6. 统一列名
# ============================================================

df.rename(columns={
    "uva": "UVA",
    "Clear sky GHI": "ClearSkyGHI"
}, inplace=True)

# ============================================================
# 7. 构造 CSI
# ============================================================

eps = 1e-6

df["CSI"] = (
    df["GHI"]
    /
    (df["ClearSkyGHI"] + eps)
)

df["CSI"] = df["CSI"].clip(0, 2)

# ============================================================
# 8. 周期时间编码
# ============================================================

print("构造时间周期特征...")

hours = df.index.hour

day_of_year = df.index.dayofyear

days_in_year = np.where(
    df.index.is_leap_year,
    366,
    365
)

# 日周期
df["hour_sin"] = np.sin(
    2 * np.pi * hours / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * hours / 24
)

# 年周期
df["day_year_sin"] = np.sin(
    2 * np.pi * day_of_year / days_in_year
)

df["day_year_cos"] = np.cos(
    2 * np.pi * day_of_year / days_in_year
)

# ============================================================
# 9. 最终检查
# ============================================================

print("\n========== FINAL DATASET ==========")

print("Shape:")
print(df.shape)

print("\nTime Range:")
print(df.index.min())
print(df.index.max())

print("\nMissing Values:")
print(df.isna().sum())

print("\nDescribe:")
print(df.describe())

# ============================================================
# 10. 保存
# ============================================================

output_file = "final_dataset_hourly.csv"

df.to_csv(output_file)

print(f"\n数据集已保存: {output_file}")
print(f"最终样本数: {len(df)}")
print(f"最终特征数: {len(df.columns)}")

读取 Weather 数据...


C:\Users\apple\AppData\Local\Temp\ipykernel_13920\3246243350.py:17: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  weather_hourly = weather.resample("1H").agg({


Weather Hourly Shape: (17477, 9)
读取 UVI 数据...


C:\Users\apple\AppData\Local\Temp\ipykernel_13920\3246243350.py:81: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .resample("1H")
C:\Users\apple\AppData\Local\Temp\ipykernel_13920\3246243350.py:99: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  clear_hourly = clear.resample("1H").mean()


UVI Hourly Shape: (23367, 1)
读取 Clear Sky 数据...
Clear Sky Shape: (26328, 1)
开始合并数据...
构造时间周期特征...

========== FINAL DATASET ==========
Shape:
(17472, 16)

Time Range:
2020-01-01 05:00:00
2021-12-29 04:00:00

Missing Values:
temp               0
rainfall           0
windspd            0
windspd_max        0
wind_d             0
GHI                0
DNI                0
DHI                0
UVA                0
UVI             6552
ClearSkyGHI        0
CSI                0
hour_sin           0
hour_cos           0
day_year_sin       0
day_year_cos       0
dtype: int64

Describe:
               temp      rainfall       windspd   windspd_max        wind_d  \
count  17472.000000  17472.000000  17472.000000  17472.000000  17472.000000   
mean      24.015100      0.248128      2.670475      6.193521    135.315880   
std        5.118763      2.052893      1.160696      2.551533     70.609965   
min        6.766667      0.000000      0.000000      0.000000      0.000000   
25%       19.951667  

In [10]:
df = pd.read_csv(
    "final_dataset_hourly.csv",
    index_col=0,
    parse_dates=True
)

print(df.shape)
print(df.head())
print(df.tail())
print(df.isna().sum().sum())

(17472, 16)
                          temp  rainfall   windspd  windspd_max     wind_d  \
time                                                                         
2020-01-01 05:00:00  16.508333       0.0  2.020000          5.4  84.966667   
2020-01-01 06:00:00  16.360000       0.1  2.413333          6.1  80.183333   
2020-01-01 07:00:00  16.391667       0.0  2.770000          6.8  82.266667   
2020-01-01 08:00:00  16.665000       0.0  2.930000          8.3  72.750000   
2020-01-01 09:00:00  17.260000       0.0  2.058333          5.6  69.600000   

                            GHI       DNI         DHI    UVA       UVI  \
time                                                                     
2020-01-01 05:00:00    0.000000  0.000000    0.000000   0.00  0.000000   
2020-01-01 06:00:00    0.000000  0.000000    0.000000   0.00  0.000000   
2020-01-01 07:00:00    1.216667  0.000000    1.633333   0.25  0.002500   
2020-01-01 08:00:00   26.316667  0.000000   25.133333   2.25  0.123333 

In [18]:
import pandas as pd
import numpy as np

# ==========================================================
# 1. Weather 数据
# ==========================================================

print("Loading weather data...")

weather = pd.read_csv("weather data and radiation.csv")

weather["time"] = pd.to_datetime(weather["time"])

weather = weather.set_index("time")

weather_hourly = weather.resample("1H").agg({
    "temp": "mean",
    "rainfall": "sum",
    "windspd": "mean",
    "windspd_max": "max",
    "wind_d": "mean",
    "GHI": "mean",
    "DNI": "mean",
    "DHI": "mean",
    "uva": "mean"
})

# 时间线性插值
weather_hourly = weather_hourly.interpolate(
    method="time",
    limit_direction="both"
)

print("Weather:")
print(weather_hourly.shape)

# ==========================================================
# 2. UVI 数据
# ==========================================================

# ==========================================================
# 2. UVI 数据
# ==========================================================

print("Loading UVI data...")

uvi = pd.read_csv(
    "uv_5min.txt",
    sep=r"\s+",
    skiprows=1,
    names=["time", "UVI"]
)

uvi["time"] = pd.to_datetime(
    uvi["time"].astype(str),
    format="%Y%m%d%H%M"
)

uvi["UVI"] = pd.to_numeric(
    uvi["UVI"],
    errors="coerce"
)

# ----------------------------------------------------------
# 异常值处理
# ----------------------------------------------------------

uvi.loc[
    uvi["UVI"] > 20,
    "UVI"
] = np.nan

print("异常值数量:")
print(uvi["UVI"].isna().sum())

uvi = uvi.set_index("time")

# ----------------------------------------------------------
# 重采样到小时
# ----------------------------------------------------------

uvi_hourly = (
    uvi
    .resample("1H")
    .mean()
)

# ----------------------------------------------------------
# 小时级缺失插值
# ----------------------------------------------------------

uvi_hourly["UVI"] = (
    uvi_hourly["UVI"]
    .interpolate(
        method="time",
        limit_direction="both"
    )
)

print("UVI:")
print(uvi_hourly.shape)

print("Hourly Missing:")
print(uvi_hourly["UVI"].isna().sum())

print("Hourly Max:")
print(uvi_hourly["UVI"].max())

# ==========================================================
# 3. Clear Sky 数据
# ==========================================================

print("Loading Clear Sky data...")

clear = pd.read_csv(
    "clear_sky_ghi_hkt.csv"
)

clear["time"] = pd.to_datetime(
    clear["time"]
)

clear = clear.set_index("time")

clear_hourly = (
    clear
    .resample("1H")
    .mean()
)

# 前面缺失补0
clear_hourly["Clear sky GHI"] = (
    clear_hourly["Clear sky GHI"]
    .fillna(0)
)

print("Clear Sky:")
print(clear_hourly.shape)

# ==========================================================
# 4. 数据合并
# ==========================================================

print("Merging datasets...")

df = weather_hourly.join(
    uvi_hourly,
    how="inner"
)

df = df.join(
    clear_hourly,
    how="left"
)

df["Clear sky GHI"] = (
    df["Clear sky GHI"]
    .fillna(0)
)

# ==========================================================
# 5. 重命名
# ==========================================================

df.rename(
    columns={
        "uva": "UVA",
        "Clear sky GHI": "ClearSkyGHI"
    },
    inplace=True
)

# ==========================================================
# 6. CSI
# ==========================================================

eps = 1e-6

df["CSI"] = (
    df["GHI"] /
    (df["ClearSkyGHI"] + eps)
)

df["CSI"] = df["CSI"].clip(
    lower=0,
    upper=2
)

# ==========================================================
# 7. 周期时间编码
# ==========================================================

print("Generating cyclical features...")

hours = df.index.hour

day_of_year = df.index.dayofyear

days_in_year = np.where(
    df.index.is_leap_year,
    366,
    365
)

df["hour_sin"] = np.sin(
    2 * np.pi * hours / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * hours / 24
)

df["day_year_sin"] = np.sin(
    2 * np.pi * day_of_year / days_in_year
)

df["day_year_cos"] = np.cos(
    2 * np.pi * day_of_year / days_in_year
)

# ==========================================================
# 8. 最终检查
# ==========================================================

print("\n========== FINAL DATASET ==========")

print("Shape:")
print(df.shape)

print("\nTime Range:")
print(df.index.min())
print(df.index.max())

print("\nMissing:")
print(df.isna().sum())

print("\nTotal Missing:")
print(df.isna().sum().sum())

print("\nUVI Statistics:")
print(df["UVI"].describe())

print("\nCSI Statistics:")
print(df["CSI"].describe())

# ==========================================================
# 9. 保存
# ==========================================================

output_file = "final_dataset_hourly.csv"

df.to_csv(output_file)

print("\nSaved:")
print(output_file)

Loading weather data...


C:\Users\apple\AppData\Local\Temp\ipykernel_13920\3650967111.py:16: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  weather_hourly = weather.resample("1H").agg({


Weather:
(17477, 9)
Loading UVI data...
异常值数量:
745
UVI:
(23367, 1)
Hourly Missing:
0
Hourly Max:
12.31
Loading Clear Sky data...
Clear Sky:
(26328, 1)
Merging datasets...
Generating cyclical features...

========== FINAL DATASET ==========
Shape:
(17472, 16)

Time Range:
2020-01-01 05:00:00
2021-12-29 04:00:00

Missing:
temp            0
rainfall        0
windspd         0
windspd_max     0
wind_d          0
GHI             0
DNI             0
DHI             0
UVA             0
UVI             0
ClearSkyGHI     0
CSI             0
hour_sin        0
hour_cos        0
day_year_sin    0
day_year_cos    0
dtype: int64

Total Missing:
0

UVI Statistics:
count    17472.000000
mean         1.361160
std          2.345591
min          0.000000
25%          0.000000
50%          0.007500
75%          1.858750
max         12.310000
Name: UVI, dtype: float64

CSI Statistics:
count    17472.000000
mean         0.523289
std          0.683023
min          0.000000
25%          0.000000
50%          

C:\Users\apple\AppData\Local\Temp\ipykernel_13920\3650967111.py:84: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .resample("1H")
C:\Users\apple\AppData\Local\Temp\ipykernel_13920\3650967111.py:127: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .resample("1H")



Saved:
final_dataset_hourly.csv
